In [1]:
from pathlib import Path
import pandas as pd

# ============================================================
# NB_
# 01. INPUT DISCOVERY
# ============================================================

ROOT = Path(r"C:\ds4b\C2011-13 Wind Farm_v4")

for folder_name in ["05_Outputs", "06_Outputs", "07_Outputs"]:
    folder = ROOT / folder_name

    print(f"\n{'=' * 65}")
    print(f"{folder_name}")
    print("=" * 65)

    if not folder.exists():
        print("Carpeta no encontrada")
        continue

    for file in sorted(folder.iterdir()):
        if file.suffix.lower() in [".xlsx", ".csv"]:
            print(f"{file.name} ({file.stat().st_size:,} bytes)")

# ------------------------------------------------------------
# MAESTRO DE ACTIVIDADES: estructura económica
# ------------------------------------------------------------

master_path = ROOT / "05_Outputs" / "Activity_Master_with_Materials.xlsx"

if master_path.exists():

    xls = pd.ExcelFile(master_path)

    print("\n" + "=" * 65)
    print("ACTIVITY MASTER — SHEETS")
    print("=" * 65)

    print(xls.sheet_names)

    for sheet in xls.sheet_names:
        df = pd.read_excel(master_path, sheet_name=sheet)

        print(f"\nHoja: {sheet}")
        print(f"Filas: {len(df):,}")
        print("Columnas:")
        print(df.columns.tolist())

        economic_cols = [
            c for c in df.columns
            if any(k in c.lower() for k in
                   ["cost", "coste", "budget", "amount", "value", "price"])
        ]

        if economic_cols:
            print("Columnas económicas:", economic_cols)

else:
    print("\nNo se encuentra Activity_Master_with_Materials.xlsx")

In [2]:
# ============================================================
# NB_11 — ECONOMIC DATA AUDIT
# 02. COST STRUCTURE & DECISION DATA
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. ACTIVITY MASTER
# ------------------------------------------------------------

master_path = ROOT / "05_Outputs" / "Activity_Master_with_Materials.xlsx"

df_master = pd.read_excel(master_path)

cost_cols = [
    "Total_Cost_Plan",
    "Fixed_Cost",
    "Variable_Cost",
    "Resource_Cost"
]

print("=" * 70)
print("1. PROJECT COST STRUCTURE")
print("=" * 70)

for col in cost_cols:
    df_master[col] = pd.to_numeric(df_master[col], errors="coerce")

print("\nDescriptive statistics:")
print(df_master[cost_cols].describe().round(2).to_string())

print("\nTotal amounts:")
print(df_master[cost_cols].sum().round(2).to_string())

print("\nMissing values:")
print(df_master[cost_cols].isna().sum().to_string())

print("\nTotal_Cost_Plan vs cost components:")
print(
    pd.Series({
        "Total_Cost_Plan": df_master["Total_Cost_Plan"].sum(),
        "Fixed + Variable + Resource":
            df_master[["Fixed_Cost", "Variable_Cost", "Resource_Cost"]]
            .sum().sum()
    }).round(2).to_string()
)

# ------------------------------------------------------------
# 2. RELEVANT ACTIVITIES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. RELEVANT ACTIVITIES — IDs 20 AND 79")
print("=" * 70)

relevant_cols = [
    "ID", "Name", "Duration_Plan_Days",
    "Total_Cost_Plan", "Fixed_Cost",
    "Variable_Cost", "Resource_Cost",
    "Material_Type", "Supply_Delay_Days",
    "Buffer_Days", "Material_Status"
]

print(
    df_master.loc[
        df_master["ID"].isin([20, 79]),
        [c for c in relevant_cols if c in df_master.columns]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 3. NB_06 DECISION POINTS
# ------------------------------------------------------------

decision_path = ROOT / "06_Outputs" / "06_Decision_Points_Base.xlsx"

print("\n" + "=" * 70)
print("3. NB_06 — DECISION POINTS")
print("=" * 70)

xls_decision = pd.ExcelFile(decision_path)

print("Sheets:", xls_decision.sheet_names)

for sheet in xls_decision.sheet_names:

    df = pd.read_excel(decision_path, sheet_name=sheet)

    print(f"\n--- {sheet} ---")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    economic_cols = [
        c for c in df.columns
        if any(k in c.lower() for k in
               ["cost", "coste", "budget", "amount", "value",
                "price", "ev", "pv", "ac", "eac", "bac"])
    ]

    if economic_cols:
        print("\nEconomic columns:")
        print(df[economic_cols].head(10).to_string(index=False))

# ------------------------------------------------------------
# 4. NB_10 ACTION EVALUATION STRUCTURE
# ------------------------------------------------------------

evaluation_path = ROOT / "07_Outputs" / "Corrective_Action_Evaluation.xlsx"

print("\n" + "=" * 70)
print("4. NB_10 — CORRECTIVE ACTION OUTPUTS")
print("=" * 70)

xls_actions = pd.ExcelFile(evaluation_path)

for sheet in xls_actions.sheet_names:

    df = pd.read_excel(evaluation_path, sheet_name=sheet)

    print(f"\n--- {sheet} ---")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

    if sheet in ["Corrective Actions", "Decision Summary"]:
        print(df.head(10).to_string(index=False))

print("\n" + "=" * 70)
print("ECONOMIC DATA AUDIT COMPLETED")
print("=" * 70)

In [3]:
# ============================================================
# NB_11 — ECONOMIC RECONCILIATION
# 03. WBS COST STRUCTURE
# ============================================================

import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. LOAD NB_06 ACTIVITY MASTER
# ------------------------------------------------------------

decision_path = ROOT / "06_Outputs" / "06_Decision_Points_Base.xlsx"

df_wbs = pd.read_excel(
    decision_path,
    sheet_name="Activity_Master"
)

print("=" * 70)
print("1. WBS STRUCTURE")
print("=" * 70)

print("Rows:", len(df_wbs))
print("\nElement types:")
print(df_wbs["Element_Type"].value_counts(dropna=False).to_string())

print("\nTerminal status:")
print(df_wbs["Is_Terminal"].value_counts(dropna=False).to_string())

print("\nExecutable status:")
print(df_wbs["Is_Executable"].value_counts(dropna=False).to_string())

# ------------------------------------------------------------
# 2. COST RECONCILIATION BY WBS LEVEL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. COSTS BY WBS LEVEL")
print("=" * 70)

wbs_summary = (
    df_wbs.groupby("WBS_Level", dropna=False)
    .agg(
        Rows=("ID", "count"),
        Total_Cost=("Total_Cost_Plan", "sum"),
        Fixed_Cost=("Fixed_Cost", "sum"),
        Variable_Cost=("Variable_Cost", "sum"),
        Resource_Cost=("Resource_Cost", "sum")
    )
    .round(2)
)

print(wbs_summary.to_string())

# ------------------------------------------------------------
# 3. TERMINAL / EXECUTABLE COST BASE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3. TERMINAL AND EXECUTABLE COSTS")
print("=" * 70)

for col in ["Is_Terminal", "Is_Executable"]:
    subset = df_wbs[df_wbs[col] == True]

    print(f"\n{col} = True")
    print("Rows:", len(subset))
    print("Total_Cost_Plan:", round(subset["Total_Cost_Plan"].sum(), 2))
    print("Fixed_Cost:", round(subset["Fixed_Cost"].sum(), 2))
    print("Variable_Cost:", round(subset["Variable_Cost"].sum(), 2))
    print("Resource_Cost:", round(subset["Resource_Cost"].sum(), 2))

# ------------------------------------------------------------
# 4. COST COMPONENT RECONCILIATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("4. COMPONENT RECONCILIATION")
print("=" * 70)

for label, subset in [
    ("All rows", df_wbs),
    ("Terminal", df_wbs[df_wbs["Is_Terminal"] == True]),
    ("Executable", df_wbs[df_wbs["Is_Executable"] == True])
]:

    components = subset[
        ["Fixed_Cost", "Variable_Cost", "Resource_Cost"]
    ].sum(min_count=1).sum()

    total = subset["Total_Cost_Plan"].sum()

    print(
        f"{label}: Total={total:,.2f} | "
        f"Components={components:,.2f} | "
        f"Difference={total-components:,.2f}"
    )

# ------------------------------------------------------------
# 5. TARGET ACTIVITIES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("5. TARGET ACTIVITIES — CA01 / CA02")
print("=" * 70)

target_cols = [
    "ID", "Name", "WBS", "WBS_Level",
    "Element_Type", "Is_Terminal", "Is_Executable",
    "Duration_Plan_Days", "Total_Cost_Plan",
    "Fixed_Cost", "Variable_Cost", "Resource_Cost"
]

print(
    df_wbs.loc[
        df_wbs["ID"].isin([20, 79]),
        [c for c in target_cols if c in df_wbs.columns]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 6. PROJECT BUDGET REFERENCE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("6. PROJECT BUDGET REFERENCE")
print("=" * 70)

root_rows = df_wbs[df_wbs["WBS_Level"] == df_wbs["WBS_Level"].min()]

print("Root-level budget:")
print(root_rows[["ID", "Name", "Total_Cost_Plan"]].to_string(index=False))

print("\nMaximum Total_Cost_Plan:", 
      round(df_wbs["Total_Cost_Plan"].max(), 2))

print("\nECONOMIC RECONCILIATION COMPLETED")

CA01 – Activity 79: coste planificado de 132.163 €, con 35 días de retraso de suministro y 21 días de buffer.



CA02 – Activity 20: coste planificado de 1.111.352 €, duración de 14 días y actividad crítica.

1. Cómo vamos a valorar las dos acciones
   
Mi propuesta es utilizar dos modelos económicos diferentes, porque los problemas son diferentes.

CA02 — Recuperación de ejecución
Aquí sí podemos calcular el valor económico de recuperar cinco días de una actividad crítica.

El modelo tendrá dos bloques:

Coste de no intervenir
- Gastos generales de obra que se prolongan.
- Supervisión y gestión del proyecto.
- Alquiler de maquinaria o equipos que deban permanecer disponibles.
- Costes financieros o contractuales, cuando proceda.
Coste de intervenir
- Horas extraordinarias.
- Segundo turno o refuerzo de personal.
- Subcontratación adicional.
- Medios auxiliares o maquinaria extraordinaria.
La lógica será:

\[ Beneficio neto = Coste evitado por recuperar 5 días- Coste de recuperación\]

CA01 — Aceleración del suministro
Aquí no vamos a fingir que diez días recuperados equivalen a diez días de proyecto.

Evaluaremos:
- Sobrecoste de transporte urgente.
- Recargo del proveedor.
- Manipulación y logística extraordinaria.
- Posible espera de equipos o personal.
- Coste de reprogramación y pérdida de productividad.
Y una cuestión importante: después de CA01 todavía quedan 19 días de exceso sobre el buffer. Por tanto, la acción reduce el riesgo, pero no lo elimina.

2. Los tres escenarios económicos
   
Para cada acción construiremos una estimación a tres puntos:

| Escenario | Interpretación |
| :--- | :--- |
| **Optimista** | Condiciones favorables, intervención eficiente y pocas incidencias adicionales. |
| **Realista** | Supuestos centrales razonables. |
| **Pesimista** | Mayor coste de intervención y consecuencias operativas más severas. |

No vamos a aplicar simplemente un +10 %, +20 % y +30 %. Cada escenario tendrá sus propios componentes y justificación.

3. Comprobacion de costes previa

In [6]:
# ============================================================
# NB_11 — TARGET ACTIVITY COST TRACE
# 04. COST HISTORY AND EVM REFERENCE
# ============================================================

decision_path = ROOT / "06_Outputs" / "06_Decision_Points_Base.xlsx"

df_perf = pd.read_excel(
    decision_path,
    sheet_name="Activity_Performance"
)

df_evm = pd.read_excel(
    decision_path,
    sheet_name="Project_EVM"
)

# ------------------------------------------------------------
# 1. TARGET ACTIVITY COST HISTORY
# ------------------------------------------------------------

print("=" * 70)
print("1. COST HISTORY — ACTIVITIES 20 AND 79")
print("=" * 70)

for activity_id in [20, 79]:

    df_target = (
        df_perf[df_perf["Activity_ID"] == activity_id]
        .sort_values("Status_Date")
    )

    print(f"\nACTIVITY {activity_id}")
    print("-" * 50)

    if df_target.empty:
        print("No records found")
        continue

    cols = [
        "TP", "Status_Date", "Pct_Complete",
        "EV", "PV", "Actual_Cost",
        "Schedule_Variance", "Cost_Variance",
        "CPI_Activity"
    ]

    print(
        df_target[
            [c for c in cols if c in df_target.columns]
        ].to_string(index=False)
    )

    print("\nTotals:")
    print(
        df_target[
            ["EV", "PV", "Actual_Cost"]
        ].sum().round(2).to_string()
    )

# ------------------------------------------------------------
# 2. PROJECT EVM — FINAL RECORD
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. PROJECT EVM — FINAL STATUS")
print("=" * 70)

df_evm["Status_Date"] = pd.to_datetime(
    df_evm["Status_Date"], errors="coerce"
)

last_evm = df_evm.sort_values("Status_Date").tail(1)

evm_cols = [
    "Status_Date", "PV", "EV", "AC",
    "SPI", "CPI", "EAC_PF1", "EAC_CPI",
    "Alarm_Level", "Alarms"
]

print(
    last_evm[
        [c for c in evm_cols if c in last_evm.columns]
    ].to_string(index=False)
)

# ------------------------------------------------------------
# 3. PROJECT COST BASE
# ------------------------------------------------------------

budget = df_wbs.loc[
    df_wbs["Is_Terminal"] == True,
    "Total_Cost_Plan"
].sum()

print("\n" + "=" * 70)
print("3. RECONCILED PROJECT BUDGET")
print("=" * 70)

print(f"Terminal activity budget: €{budget:,.2f}")

print("\nAUDIT COMPLETED")